In [1]:
import torch   # ← MUST come first

# ========================================
# DEVICE CONFIGURATION (MPS / CUDA / CPU)
# ========================================

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Using Apple MPS GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("⚠️ Using CPU")

🚀 Using Apple MPS GPU


In [2]:
import os, json, rasterio, glob
import numpy as np
from rasterio.windows import Window
from rasterio import features
from shapely.geometry import Polygon

# Configuration
RAW_DIR = '../raw_data'   
OUT_DIR = '../train_data'   
TILE_SIZE = 512
OVERLAP = 128            

# ========================================
# SELECTED TRAIN + TEST SCENES ONLY
# ========================================

allowed_scenes = [
    # TRAIN - Laura
    "laura-01",
    "laura-02",

    # TRAIN - Ian
    *[f"1001-Ian-{str(i).zfill(2)}" for i in range(1, 12)],
    *[f"1002-Ian-{str(i).zfill(2)}" for i in range(1, 4)],

    # TEST - Michael
    "Michael-01",
    "Michael-02",

    # TEST - Idalia
    "Idalia-01",
    "Idalia-02"
]


def process_disaster_scenes():
    os.makedirs(os.path.join(OUT_DIR, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, "masks"), exist_ok=True)
    
    stride = TILE_SIZE - OVERLAP 
    
    # Get ALL tif files
    all_tifs = glob.glob(os.path.join(RAW_DIR, "*.tif"))

    # Filter only allowed scenes
    tif_files = [
        tif for tif in all_tifs
        if any(scene in os.path.basename(tif) for scene in allowed_scenes)
    ]
    
    print(f"🔍 Found {len(tif_files)} selected TIFF images.")

    for tif_path in tif_files:
        filename = os.path.basename(tif_path)
        file_prefix = filename.replace('.tif', '').replace('_tif', '')
        
        base_json = os.path.join(RAW_DIR, f"{file_prefix}_json.json")
        align_json = os.path.join(RAW_DIR, f"{file_prefix}_json_aligned.json")
        
        if not os.path.exists(base_json) or not os.path.exists(align_json):
            print(f"⚠️ Skipping {file_prefix}: Missing JSON files.")
            continue

        print(f"🏗️ Processing Scene: {file_prefix}")

        with open(align_json) as f:
            align_data = json.load(f)
        s_x = np.mean([p[1][0] - p[0][0] for p in align_data])
        s_y = np.mean([p[1][1] - p[0][1] for p in align_data])
        
        with rasterio.open(tif_path) as src:
            with open(base_json) as f:
                polys = []
                for entry in json.load(f):
                    if 'pixels' in entry:
                        c = [(p['x'] + s_x, p['y'] + s_y) for p in entry['pixels']]
                        world_c = [src.transform * (px, py) for px, py in c]
                        polys.append(Polygon(world_c))

            for y in range(0, src.height - TILE_SIZE, stride):
                for x in range(0, src.width - TILE_SIZE, stride):
                    window = Window(x, y, TILE_SIZE, TILE_SIZE)
                    
                    mask = features.rasterize(
                        polys, 
                        out_shape=(TILE_SIZE, TILE_SIZE), 
                        transform=src.window_transform(window), 
                        fill=0, default_value=1, dtype=np.uint8
                    )
                    
                    if np.sum(mask) > 50:
                        tile_id = f"{file_prefix}_y{y}_x{x}.npy"
                        img_patch = src.read([1, 2, 3], window=window)
                        
                        np.save(os.path.join(OUT_DIR, "images", tile_id),
                                img_patch.transpose(1, 2, 0))
                        np.save(os.path.join(OUT_DIR, "masks", tile_id),
                                mask)

    total_images = len(os.listdir(os.path.join(OUT_DIR, "images")))
    print(f"✅ Preprocessing Complete! Generated {total_images} tiles.")


# Run preprocessing
process_disaster_scenes()


# ========================================
# DATASET + TRAIN/TEST SPLIT
# ========================================

import torch
from torch.utils.data import Dataset, Subset, DataLoader

class DisasterFoundationDataset(Dataset):
    def __init__(self, root_dir='../train_data'):
        self.img_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')
        self.filenames = sorted(
            [f for f in os.listdir(self.img_dir) if f.endswith('.npy')]
        )

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.load(os.path.join(self.img_dir, fname)).astype(np.float32) / 255.0
        mask = np.load(os.path.join(self.mask_dir, fname)).astype(np.float32)
        
        img_tensor = torch.from_numpy(img).permute(2, 0, 1)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        return img_tensor, mask_tensor


# Initialize dataset
full_dataset = DisasterFoundationDataset()


# -------------------------------
# TRAIN / TEST SPLIT
# -------------------------------

train_targets = [
    "laura-01",
    "laura-02",
    *[f"1001-Ian-{str(i).zfill(2)}" for i in range(1, 12)],
    *[f"1002-Ian-{str(i).zfill(2)}" for i in range(1, 4)],
]

test_targets = [
    "Michael-01",
    "Michael-02",
    "Idalia-01",
    "Idalia-02"
]

train_indices = []
test_indices = []

for i, fname in enumerate(full_dataset.filenames):
    if any(target in fname for target in train_targets):
        train_indices.append(i)
    elif any(target in fname for target in test_targets):
        test_indices.append(i)


train_loader = DataLoader(
    Subset(full_dataset, train_indices),
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    Subset(full_dataset, test_indices),
    batch_size=1,
    shuffle=False
)

print("📊 SPLIT SUMMARY:")
print(f"🏠 Training Tiles: {len(train_indices)}")
print(f"🔍 Testing Tiles : {len(test_indices)}")

🔍 Found 20 selected TIFF images.
🏗️ Processing Scene: 1001-Ian-03
🏗️ Processing Scene: 1001-Ian-02
🏗️ Processing Scene: laura-01
🏗️ Processing Scene: 1002-Ian-01
🏗️ Processing Scene: 1001-Ian-10
🏗️ Processing Scene: 1002-Ian-03
🏗️ Processing Scene: 1001-Ian-09
🏗️ Processing Scene: 1002-Ian-02
🏗️ Processing Scene: 1001-Ian-08
🏗️ Processing Scene: laura-02
🏗️ Processing Scene: 1001-Ian-01
🏗️ Processing Scene: 1001-Ian-11
🏗️ Processing Scene: 1001-Ian-04
🏗️ Processing Scene: Michael-02
🏗️ Processing Scene: 1001-Ian-05
🏗️ Processing Scene: Idalia-01
🏗️ Processing Scene: 1001-Ian-07
🏗️ Processing Scene: 1001-Ian-06
🏗️ Processing Scene: Idalia-02
🏗️ Processing Scene: Michael-01
✅ Preprocessing Complete! Generated 67631 tiles.
📊 SPLIT SUMMARY:
🏠 Training Tiles: 51127
🔍 Testing Tiles : 14106


In [3]:
import os, time, torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, Subset, DataLoader
from tqdm import tqdm
import numpy as np

# --- 1. SYSTEM & DEVICE ---
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🚀 Hardware Confirmed! Training on: {device.type.upper()}")

# --- 2. ADVANCED AUGMENTATION ---
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=20, val_shift_limit=15, p=1),
    ], p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# --- 3. DATASET CLASS ---
class DisasterDataset(Dataset):
    def __init__(self, root_dir='../train_data', transform=None):
        self.img_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')
        self.filenames = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.npy')])
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.load(os.path.join(self.img_dir, fname))
        mask = np.load(os.path.join(self.mask_dir, fname))
        
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented['image'], augmented['mask']
        
        return img, mask.unsqueeze(0).float()

# --- 4. DATA SPLIT LOGIC ---
# Training: Laura and all 14 Ian Scenes | Testing: Michael and Idalia Scenes
full_train_ds = DisasterDataset(root_dir='../train_data', transform=train_transform)
full_val_ds = DisasterDataset(root_dir='../train_data', transform=val_transform)

train_targets = ["laura-01", "laura-02"] + [f"1001-Ian-{str(i).zfill(2)}" for i in range(1, 12)] + [f"1002-Ian-{str(i).zfill(2)}" for i in range(1, 4)]
test_targets = ["Michael-01", "Michael-02", "Idalia-01", "Idalia-02"]

train_indices = [i for i, f in enumerate(full_train_ds.filenames) if any(t in f for t in train_targets)]
test_indices = [i for i, f in enumerate(full_val_ds.filenames) if any(t in f for t in test_targets)]

train_loader = DataLoader(Subset(full_train_ds, train_indices), batch_size=12, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(Subset(full_val_ds, test_indices), batch_size=1, num_workers=0)

print(f"📊 Dataset Ready! Training: {len(train_indices)} tiles | Test: {len(test_indices)} tiles")

# --- 5. CUSTOM FOCAL TVERSKY LOSS ---
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75):
        super(FocalTverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

    def forward(self, inputs, targets, smooth=1e-6):
        # Flatten label and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        # True Positives, False Positives & False Negatives
        TP = (inputs * targets).sum()    
        FP = ((1 - targets) * inputs).sum()
        FN = (targets * (1 - inputs)).sum()
       
        tversky = (TP + smooth) / (TP + self.alpha * FP + self.beta * FN + smooth)  
        focal_tversky = torch.pow((1 - tversky), self.gamma)
        
        return focal_tversky

# --- 6. ARCHITECTURE & OPTIMIZATION ---
model = smp.Unet(
    encoder_name="resnet50", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1, 
    decoder_attention_type='scse', 
    activation='sigmoid'
).to(device)

criterion = FocalTverskyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)

# --- 7. TRAINING LOOP ---
best_iou = 0.0
patience = 5
no_improve = 0

for epoch in range(30):
    start_time = time.time()
    model.train()
    running_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/30 [Train]")
    
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()
        pbar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
    
    scheduler.step()
    
    # Validation Phase
    model.eval()
    running_test_loss = 0
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            running_test_loss += criterion(outputs, masks).item()
            
            preds = (outputs > 0.5).float()
            tp += (preds * masks).sum().item()
            fp += (preds * (1 - masks)).sum().item()
            fn += ((1 - preds) * masks).sum().item()
            
    iou = tp / (tp + fp + fn + 1e-7)
    duration = time.time() - start_time
    
    status = ""
    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), "best_resnet50_hurricane.pth")
        status = "🌟 NEW BEST!"
        no_improve = 0
    else:
        no_improve += 1
        status = f"⏳ (Patience: {no_improve}/{patience})"
        
    print(f"\n✅ Epoch {epoch+1} Results:")
    print(f"   🔹 Train Loss: {running_train_loss/len(train_loader):.4f} | Test Loss: {running_test_loss/len(test_loader):.4f}")
    print(f"   🔹 Test IoU  : {iou:.4f} | Time: {duration:.1f}s {status}\n")

    if no_improve >= patience:
        print("🛑 Early stopping triggered. Generalization limit reached.")
        break

print(f"🏁 Final Complete. Global Best IoU on Michael & Idalia: {best_iou:.4f}")

🚀 Hardware Confirmed! Training on: MPS
📊 Dataset Ready! Training: 51127 tiles | Test: 14106 tiles


Epoch 1/30 [Train]:   0%|                                                                      | 0/4261 [00:00<?, ?it/s]/Users/shashank/miniconda3/envs/wlstm/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Epoch 1/30 [Train]: 100%|████████████████████████████████████████| 4261/4261 [51:45<00:00,  1.37it/s, batch_loss=0.6524]



✅ Epoch 1 Results:
   🔹 Train Loss: 0.2932 | Test Loss: 0.5409
   🔹 Test IoU  : 0.4772 | Time: 3901.1s 🌟 NEW BEST!



Epoch 2/30 [Train]: 100%|██████████████████████████████████████| 4261/4261 [1:00:58<00:00,  1.16it/s, batch_loss=0.2339]



✅ Epoch 2 Results:
   🔹 Train Loss: 0.2588 | Test Loss: 0.5402
   🔹 Test IoU  : 0.5058 | Time: 4228.9s 🌟 NEW BEST!



...


✅ Epoch 5 Results:
   🔹 Train Loss: 0.2372 | Test Loss: 0.5399
   🔹 Test IoU  : 0.4712 | Time: 4213.6s ⏳ (Patience: 3/5)



Epoch 6/30 [Train]: 100%|██████████████████████████████████████| 4261/4261 [1:05:19<00:00,  1.09it/s, batch_loss=0.1412]



✅ Epoch 6 Results:
   🔹 Train Loss: 0.2305 | Test Loss: 0.5228
   🔹 Test IoU  : 0.4928 | Time: 4703.6s ⏳ (Patience: 4/5)



Epoch 7/30 [Train]: 100%|████████████████████████████████████████| 4261/4261 [57:00<00:00,  1.25it/s, batch_loss=0.2475]



✅ Epoch 7 Results:
   🔹 Train Loss: 0.2249 | Test Loss: 0.5193
   🔹 Test IoU  : 0.4958 | Time: 4202.8s ⏳ (Patience: 5/5)

🛑 Early stopping triggered. Generalization limit reached.
🏁 Final Complete. Global Best IoU on Michael & Idalia: 0.5058


In [1]:
import os, torch
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, Subset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ----------------------------
# 1) DEVICE
# ----------------------------
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🚀 Evaluating on: {device.type.upper()}")

# ----------------------------
# 2) VAL/TEST TRANSFORM (same as yours)
# ----------------------------
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ----------------------------
# 3) DATASET (same as yours)
# ----------------------------
class DisasterDataset(Dataset):
    def __init__(self, root_dir='../train_data', transform=None):
        self.img_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')
        self.filenames = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.npy')])
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.load(os.path.join(self.img_dir, fname))
        mask = np.load(os.path.join(self.mask_dir, fname))

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]

        return img, mask.unsqueeze(0).float()

# ----------------------------
# 4) BUILD TEST LOADER (edit targets if needed)
# ----------------------------
full_ds = DisasterDataset(root_dir="../train_data", transform=val_transform)

# Example: your earlier test split (Michael + Idalia)
test_targets = ["Michael-01", "Michael-02", "Idalia-01", "Idalia-02"]
test_indices = [i for i, f in enumerate(full_ds.filenames) if any(t in f for t in test_targets)]
test_loader = DataLoader(Subset(full_ds, test_indices), batch_size=1, shuffle=False, num_workers=0)

print(f"📊 Test tiles: {len(test_indices)}")

# ----------------------------
# 5) LOAD BEST MODEL (same architecture)
# ----------------------------
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_attention_type="scse",
    activation="sigmoid"
).to(device)

ckpt_path = "best_resnet50_hurricane.pth"
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state, strict=True)
model.eval()

# ----------------------------
# 6) METRICS
# ----------------------------
def eval_metrics(model, loader, threshold=0.5):
    tp = fp = fn = tn = 0.0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            probs = model(imgs)
            preds = (probs > threshold).float()

            tp += (preds * masks).sum().item()
            fp += (preds * (1 - masks)).sum().item()
            fn += ((1 - preds) * masks).sum().item()
            tn += ((1 - preds) * (1 - masks)).sum().item()

    eps = 1e-7
    iou = tp / (tp + fp + fn + eps)
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)          # Recall (Target)
    f1 = (2 * precision * recall) / (precision + recall + eps)
    specificity = tn / (tn + fp + eps)
    balanced_acc = 0.5 * (recall + specificity)
    overall_acc = (tp + tn) / (tp + tn + fp + fn + eps)

    return {
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall (Target)": recall,
        "F1-score": f1,
        "Specificity": specificity,
        "Balanced Acc.": balanced_acc,
        "Overall Accuracy": overall_acc,
    }

metrics = eval_metrics(model, test_loader, threshold=0.5)

# ----------------------------
# 7) PRINT LIKE YOUR DOCUMENT
# ----------------------------
print("\n--- 🌍 GLOBAL DISASTER FOUNDATION RESULTS ---")
print(f"IoU              : {metrics['IoU']:.4f}")
print(f"Dice             : {metrics['Dice']:.4f}")
print(f"Precision        : {metrics['Precision']:.4f}")
print(f"Recall (Target)  : {metrics['Recall (Target)']:.4f}")
print(f"F1-score         : {metrics['F1-score']:.4f}")
print(f"Specificity      : {metrics['Specificity']:.4f}")
print(f"Balanced Acc.    : {metrics['Balanced Acc.']:.4f}")
print(f"Overall Accuracy : {metrics['Overall Accuracy']:.4f}")

🚀 Evaluating on: MPS
📊 Test tiles: 14106

--- 🌍 GLOBAL DISASTER FOUNDATION RESULTS ---
IoU              : 0.5058
Dice             : 0.6718
Precision        : 0.5826
Recall (Target)  : 0.7933
F1-score         : 0.6718
Specificity      : 0.7388
Balanced Acc.    : 0.7660
Overall Accuracy : 0.7559
